# Coder 에이전트 구축

Navigator 에이전트가 브라우저를 탐색하여 **크롤링 청사진(Blueprint)**을 설계하는 역할이었다면, 이번에 만들 **Coder 에이전트**는 그 청사진을 받아 **파이썬 코드를 직접 작성하고 실행하며 디버깅하는 '시니어 개발자'** 역할을 수행합니다.

단순히 코드를 짜서 글자로만 알려주는 LLM이 아니라,
**코드를 `.py` 파일로 저장하고 → 직접 실행해보고 → 에러가 나면 스스로 고치는(디버깅)** 능력을 갖춘 완벽한 자율 에이전트입니다.

### 학습 흐름

| Phase | 내용 | 핵심 학습 |
|:---|:---|:---|
| **Phase 1** | 간단한 코드 실행 에이전트를 직접 만들어보기 | 코드 실행 도구, 도구 분리, 자가 디버깅 루프 |
| **Phase 2** | 실제 `app/`의 v1 Coder 코드를 이식하며 이해하기 | 시니어 엔지니어 행동 지침, 데이터 검증, 파일 검색 미들웨어 |

.env 파일을 로드하여 환경변수에 등록합니다. 노트북을 위한 실행 경로를 세팅합니다.

In [ ]:
import os, sys
from dotenv import load_dotenv

load_dotenv(override=True)

# PROJECT_ROOT를 .env에서 읽기 (없으면 현재 디렉토리)
project_root = os.getenv("PROJECT_ROOT", os.getcwd())

# 프로젝트 루트가 유효하지 않으면, 현재 위치에서 상위로 찾기
if not os.path.exists(os.path.join(project_root, "app")):
    # 상위 디렉토리 탐색
    current = os.getcwd()
    for _ in range(5):
        if os.path.exists(os.path.join(current, "app")):
            project_root = current
            break
        current = os.path.dirname(current)

# Working Directory 설정
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import app

print(f"✅ Project Root: {project_root}")
print(f"✅ Working Directory: {os.getcwd()}")

In [ ]:
import nest_asyncio

# 주피터 노트북 환경에서 비동기 루프 중복 실행 오류를 방지하기 위해 사용합니다.
nest_asyncio.apply()

In [ ]:
os.environ["GOOGLE_API_KEY"]
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "develop_Coder_agent" 

---
## Phase 1: 직접 만들어보기

### Part 1: 출발점 — 코드 실행 도구의 탄생

LLM에게 **"코드를 실행하는 능력"**을 주려면 어떻게 해야 할까요?

핵심 아이디어는 `subprocess` 모듈을 활용하여, 지정된 격리 공간(`code_artifacts/`) 안에서 파이썬 스크립트를 **저장하고 실행**하는 전용 도구를 만들어 제공하는 것입니다.

가장 기본적인 2개의 도구로 시작합니다:
1. **`create_new_file`** — 코드를 `.py` 파일로 저장
2. **`run_python_script`** — 저장된 파일을 독립 프로세스에서 실행

이 2개만으로도 "코드를 짜고 실행하는" 에이전트를 만들 수 있습니다!

In [ ]:
import os
import subprocess
from langchain.tools import tool

# ==========================================
# 1. 🗂️ 코드 실행 격리 공간 설정
# ==========================================
# 에이전트가 생성하는 모든 파일은 이 폴더 안에서만 관리됩니다.
ARTIFACT_DIR = os.path.join(os.getenv("PROJECT_ROOT", os.getcwd()), "artifacts", "code")
os.makedirs(ARTIFACT_DIR, exist_ok=True)
print(f"📁 코드 실행 공간: {ARTIFACT_DIR}")

# ==========================================
# 2. 🛠️ 도구 1: 파일 생성
# ==========================================
@tool(parse_docstring=True)
def create_new_file(filepath: str, content: str) -> str:
    """새로운 파이썬 파일이나 텍스트 문서를 생성하고 초기 내용을 통째로 작성합니다.
    이미 같은 이름의 파일이 존재할 경우 완전히 덮어씁니다!
    
    Args:
        filepath: 생성할 파일명 (예: main.py)
        content: 파일에 들어갈 초기 파이썬 코드 전체 스크립트
    """
    safe_filepath = os.path.join(ARTIFACT_DIR, os.path.basename(filepath))
    with open(safe_filepath, "w", encoding="utf-8") as f:
        f.write(content)
    return f"[Success] '{filepath}' 파일이 성공적으로 생성되었습니다."

# ==========================================
# 3. 🛠️ 도구 2: 파이썬 스크립트 실행
# ==========================================
@tool(parse_docstring=True)
def run_python_script(filepath: str, script_args: str = "") -> str:
    """저장된 파이썬 스크립트를 즉시 독립된 프로세스에서 실행하고 그 결과(출력 및 에러 로그)를 반환합니다.
    코드를 생성한 직후에는 반드시 이 툴을 호출하여 에러 없이 돌아가는지 검증하세요.
    
    Args:
        filepath: 실행할 파이썬 파일명 (예: main.py)
        script_args: 실행 시 덧붙일 커맨드라인 인자 (선택사항)
    """
    safe_filename = os.path.basename(filepath)
    full_path = os.path.join(ARTIFACT_DIR, safe_filename)
    
    if not os.path.exists(full_path):
         return f"[Error] 실행할 파일이 존재하지 않습니다: {safe_filename}"
         
    command = ["python", safe_filename]
    if script_args:
        command.extend(script_args.split())
        
    print(f"\n🚀 [Coder Run] '{safe_filename}' 실행 중...")
    
    try:
        result = subprocess.run(
            command, 
            cwd=ARTIFACT_DIR,
            capture_output=True, 
            text=True, 
            timeout=120 
        )
        
        output = result.stdout
        if result.stderr:
            output += f"\n[Error Output]\n{result.stderr}"
            
        if not output.strip():
            output = "[System] 코드가 에러 없이 정상 실행되었으나, 터미널에 출력(print)된 내용이 없습니다."
            
        return output
        
    except subprocess.TimeoutExpired:
        return "[Error] 실행 시간(120초)을 초과했습니다. 무한 루프나 블로킹 처리를 확인하세요."
    except Exception as e:
        return f"[System Error] 코드 실행 오류 발생: {str(e)}"

#### 간단한 Coder 에이전트 v1 조립

도구 2개(`create_new_file` + `run_python_script`)와 간단한 시스템 프롬프트만으로 에이전트를 조립합니다.

> 📖 LangChain의 `create_agent`와 `@tool` 문법은 `reference/LangChain/` 폴더에 정리되어 있습니다.

In [ ]:
from dataclasses import dataclass
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

@dataclass
class SimpleCoderContext:
    pass

coder_model = init_chat_model("google_genai:gemini-flash-latest", temperature=0.2)
checkpointer = InMemorySaver()

SIMPLE_CODER_PROMPT = """
당신은 파이썬 코드를 작성하고 실행하는 코딩 비서입니다.

[행동 지침]
1. 코드를 작성할 때는 `create_new_file` 도구로 파일을 생성하세요.
2. 파일을 생성한 직후 반드시 `run_python_script`로 실행하여 결과를 확인하세요.
3. 결과에 에러가 있으면 파일을 다시 생성하여 수정하고 재실행하세요.
4. 최종 결과를 사용자에게 깔끔하게 보고하세요.
"""

simple_coder = create_agent(
    model=coder_model,
    system_prompt=SIMPLE_CODER_PROMPT,
    context_schema=SimpleCoderContext,
    tools=[create_new_file, run_python_script],
    checkpointer=checkpointer
)

print("✅ Simple Coder v1 조립 완료 (도구 2개: create + run)")

#### 🧪 테스트: "코드를 짜고 실행해줘"

Coder에게 1부터 100까지의 소수를 구하는 코드를 짜달라고 요청합니다.
과연 **직접 계산해서 답을 내는지**, 아니면 **코드를 짜서 실행시켜 보는지** 확인하세요!

In [ ]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "simple_coder_test_1"}}
context = SimpleCoderContext()

query = "1부터 100까지의 숫자 중에서 소수(Prime Number)만 찾아내는 파이썬 코드를 작성하고 실행해서, 나에게 출력 결과만 깔끔하게 보여줘."
print(f"👤 User: {query}\n")
print("-" * 50)

response = await simple_coder.ainvoke(
    {"messages": [HumanMessage(query)]},
    config=config, context=context
)

print(f"\n🤖 Assistant: {response['messages'][-1].content}")

In [ ]:
for m in response["messages"]:
    m.pretty_print()

### 💡 관찰: 에러가 나면 어떻게 하지?

위 테스트에서 에이전트는 코드를 짜고 실행하는 데 성공했습니다. 하지만 한 가지 문제가 있습니다:

**만약 코드에 에러가 있으면?** → `create_new_file`로 **전체 코드를 처음부터 다시 작성**해야 합니다.

| 문제 | 결과 |
|:---|:---|
| 100줄짜리 코드에서 3번째 줄만 틀렸는데 | 100줄 전체를 다시 작성 |
| 어디가 틀렸는지 모르는 상태에서 | 감으로 수정 시도 |
| 토큰 비용 | 매번 전체 코드를 생성하므로 비용 폭증 |

인간 개발자라면 **줄 번호를 보고 딱 그 부분만 수정**하겠죠?

> 💡 **핵심 인사이트**: "코드를 줄 번호와 함께 읽고, 특정 줄만 교체하는" 도구가 추가되면 에이전트도 인간 개발자처럼 효율적으로 디버깅할 수 있습니다.

---
### Part 2: 도구 개선 — 외과수술적 코드 편집

#### 🛠️ 도구 추가 1: `read_code_file` — 줄 번호와 함께 코드 읽기

에이전트가 기존 파일의 내용을 **줄 번호(Line number)와 함께** 읽을 수 있게 합니다.
이렇게 하면 에이전트는 "7번 줄에서 IndentationError가 발생" 같은 에러 메시지를 받았을 때,
`read_code_file`로 해당 줄을 정확히 확인할 수 있습니다.

```
001 | import os
002 | 
003 | def hello():
004 |     print("Hello, World!")
005 |     
```

> 💡 에이전트는 도구의 **return 값**만 볼 수 있습니다. 줄 번호를 포함해서 반환하면, 에이전트가 정확한 위치를 파악할 수 있게 됩니다.

In [ ]:
@tool(parse_docstring=True)
def read_code_file(filepath: str, start_line: int = 1, end_line: int = None) -> str:
    """지정된 파일의 내용을 줄 번호(Line number)와 함께 읽어옵니다.
    코드를 수정하기 전, 정확히 몇 번째 줄을 수정해야 할지 파악하기 위해 반드시 먼저 사용하세요.
    
    Args:
        filepath: 읽을 파일의 경로 (파일명만 입력하면 code_artifacts 폴더 안에서 찾습니다)
        start_line: 읽기 시작할 줄 번호 (기본값: 1)
        end_line: 읽기를 끝낼 줄 번호 (입력하지 않으면 끝까지 읽음)
    """
    safe_filepath = os.path.join(ARTIFACT_DIR, os.path.basename(filepath))
    if not os.path.exists(safe_filepath):
        return f"[Error] 파일이 존재하지 않습니다: {safe_filepath}"
        
    with open(safe_filepath, "r", encoding="utf-8") as f:
        lines = f.readlines()
        
    end = end_line if end_line else len(lines)
    start = max(1, start_line)
    
    if start > len(lines):
        return "[Error] start_line이 파일의 전체 줄 수보다 큽니다."
        
    output = []
    for i in range(start - 1, min(end, len(lines))):
        output.append(f"{i + 1:03d} | {lines[i].rstrip()}")
        
    return "\n".join(output)

#### 🛠️ 도구 추가 2: `edit_code_file` — 특정 줄만 외과수술처럼 교체

이제 핵심입니다. 파일 전체를 다시 작성하는 대신, **특정 줄 범위(start_line ~ end_line)만 새 코드로 교체**하는 도구입니다.

```
# edit_code_file("main.py", start_line=3, end_line=4, new_content="def hello():\n    print('Hi!')")

Before:                          After:
001 | import os                  001 | import os
002 |                            002 |
003 | def hello():         →     003 | def hello():
004 |     print("old")           004 |     print('Hi!')
005 |                            005 |
```

이 도구를 추가하면 에이전트의 디버깅 루프가 완성됩니다:

```
create → run → 에러 발견 → read → edit → run → 성공! ✅
```

In [ ]:
@tool(parse_docstring=True)
def edit_code_file(filepath: str, start_line: int, end_line: int, new_content: str) -> str:
    """기존 파이썬 파일의 특정 줄(Line) 구간만 새로운 내용으로 교체합니다.
    파일 전체를 덮어쓰는 대신, 수정이 필요한 좁은 범위의 코드만 외과수술처럼 변경하세요.
    
    Args:
        filepath: 수정할 기존 파일명
        start_line: 교체를 시작할 기존 줄 번호 (이 줄부터 덮어써짐)
        end_line: 교체를 끝낼 기존 줄 번호 (이 줄까지 덮어써짐)
        new_content: 해당 구간에 통째로 새로 들어갈 코드 내용
    """
    safe_filepath = os.path.join(ARTIFACT_DIR, os.path.basename(filepath))
    if not os.path.exists(safe_filepath):
        return f"[Error] 파일이 존재하지 않습니다. create_new_file을 먼저 사용하세요: {safe_filepath}"
        
    with open(safe_filepath, "r", encoding="utf-8") as f:
        lines = f.readlines()
        
    if start_line < 1 or end_line > len(lines) or start_line > end_line:
        return f"[Error] 잘못된 줄 번호 범위입니다. (현재 파일 총 라인 수: {len(lines)})"
        
    new_lines = [line + "\n" for line in new_content.split("\n")]
    updated_lines = lines[:start_line-1] + new_lines + lines[end_line:]
    
    with open(safe_filepath, "w", encoding="utf-8") as f:
        f.writelines(updated_lines)
        
    return f"[Success] {filepath} 파일의 {start_line}~{end_line} 라인이 성공적으로 교체되었습니다."

#### 개선된 Coder v2 조립

이제 **4개의 분리된 도구**를 사용하는 개선된 Coder를 조립합니다:

| 도구 | 역할 |
|:---|:---|
| `create_new_file` | 파일 최초 생성 |
| `read_code_file` | 줄 번호와 함께 코드 읽기 |
| `edit_code_file` | 특정 줄만 외과수술처럼 교체 |
| `run_python_script` | 저장된 파일 실행 |

In [ ]:
checkpointer_v2 = InMemorySaver()

CODER_V2_PROMPT = """
당신은 파이썬 코드를 작성하고 실행하는 코딩 비서입니다.

[행동 지침]
1. 코드를 처음 작성할 때는 `create_new_file`로 파일을 생성하세요.
2. 파일을 생성한 직후 반드시 `run_python_script`로 실행하여 결과를 확인하세요.
3. 기존 파일을 수정해야 할 때는 절대로 전체 코드를 다시 작성하지 마세요.
   반드시 `read_code_file`로 줄 번호를 확인한 뒤, `edit_code_file`로 해당 줄만 교체하세요.
4. 에러가 발생하면: read_code_file → 에러 줄 확인 → edit_code_file → run_python_script 순서로 디버깅하세요.
5. 최종 결과를 사용자에게 깔끔하게 보고하세요.
"""

coder_v2 = create_agent(
    model=coder_model,
    system_prompt=CODER_V2_PROMPT,
    context_schema=SimpleCoderContext,
    tools=[create_new_file, read_code_file, edit_code_file, run_python_script],
    checkpointer=checkpointer_v2
)

print("✅ Coder v2 조립 완료 (도구 4개: create + read + edit + run)")

#### 🧪 테스트: 도구 분리 + 자가 디버깅 능력 검증

3단계 미션을 통해 Coder v2가 **create → run → read → edit → run** 루프를 자율적으로 수행하는지 확인합니다.

1. `calculator.py`를 만들고 `add`, `subtract` 함수를 가진 Calculator 클래스를 작성 → 실행
2. 기존 파일의 특정 라인을 수정(`edit_code_file`)하여 `multiply`와 `divide` 함수를 추가 → **전체 파일을 다시 쓰면 안 됨!**
3. 수정된 파일을 다시 실행하고 결과 보고

In [ ]:
config_v2 = {"configurable": {"thread_id": "coder_v2_surgical"}}
context_v2 = SimpleCoderContext()

# 3단계 미션: 생성 → 디버깅 → 기능 확장
query = (
    "1. 'calculator.py'를 파이썬으로 만들어줘. "
    "add, subtract 함수를 가진 Calculator 클래스를 짜고 "
    "출력으로 'Calculator Created'를 찍게 한 뒤 실행해봐.\n"
    "2. 실행을 확인한 뒤에는 calculator.py의 특정 라인을 수정(edit_code_file 사용)해서 "
    "multiply와 divide 함수를 추가해 봐. 전체 파일을 새로 덮어쓰면 절대 안 돼!\n"
    "3. 마지막으로 수정된 파일을 다시 한번 실행해보고 오류가 없으면 결과를 보고해줘."
)
print(f"👤 User Mission:\n{query}\n")
print("-" * 50)

response = await coder_v2.ainvoke(
    {"messages": [HumanMessage(query)]},
    config=config_v2, context=context_v2
)

print(f"\n🤖 Assistant: {response['messages'][-1].content}")

In [ ]:
for m in response["messages"]:
    m.pretty_print()

이어서 멀티턴으로 결과물을 확인해봅시다.

In [ ]:
query2 = "좋아, 그럼 방금 생성한 코드 파일 안에 무슨 내용이 적혀있는지 정확히 확인하고 알려줄래?"
print(f"👤 User: {query2}\n")
print("-" * 50)

response2 = await coder_v2.ainvoke(
    {"messages": [HumanMessage(query2)]},
    config=config_v2, context=context_v2
)
print(f"\n🤖 Assistant: {response2['messages'][-1].content}")

In [ ]:
for m in response2["messages"]:
    m.pretty_print()

#### 🎯 Phase 1 정리

| 항목 | 우리가 만든 에이전트 |
|:---|:---|
| 도구 | `create_new_file`, `read_code_file`, `edit_code_file`, `run_python_script` (4개) |
| 프롬프트 | 5줄짜리 간단 지침 |
| 디버깅 | create → run → read → edit → run 자동 루프 |
| 출력 | 자유 텍스트 |

이 에이전트는 **코드를 짜고 스스로 디버깅하는 자율 코딩 에이전트**로 이미 유용합니다.
하지만 **시니어 엔지니어** 수준으로 올라가려면 몇 가지가 부족합니다:

| 한계 | 설명 |
|:---|:---|
| **데이터 검증 없음** | JSON 수집 결과가 올바른지 자동 확인 불가 |
| **스크래핑 전용 지침 없음** | Blueprint 해석, rendering_type 판단 등의 전문 지식 부재 |
| **파일 검색 불가** | 작업 폴더에 어떤 파일들이 있는지 파악 불가 |
| **점진적 수집 전략 없음** | 10건 테스트 → 전체 확대 패턴 미적용 |
| **에스컬레이션 규칙 없음** | 같은 에러를 무한 반복할 위험 |

> **Phase 2**에서는 `app/`에 구축된 실제 Coder v1 코드를 살펴보며, 이 한계를 어떻게 극복했는지 이해합니다.

---
## Phase 2: v1 코드 이식 — 실제 Coder의 내부 구조

### Part 3: Coder의 핵심 설계 철학 — 시니어 엔지니어의 7가지 원칙

실제 프로덕션 Coder는 Phase 1의 간단한 에이전트와 **설계 철학 자체가 다릅니다**.
단순히 코드를 짜는 것이 아니라, **시니어 소프트웨어 엔지니어의 사고방식**을 프롬프트에 녹여넣었습니다.

| # | 원칙 | Phase 1에서 체험 |
|:---:|:---|:---:|
| 1 | 분리된 도구의 영리한 사용 | ✅ 4개 도구 분리 |
| 2 | 외과수술적 수정 (Surgical Edit) | ✅ read → edit 루프 |
| 3 | 검증 없는 코딩은 없다 (Test-Driven) | ⬜ run만 실행, 데이터 검증은 없었음 |
| 4 | 젠틀한 소통 | ⬜ |
| 5 | 코드 품질 (PEP 8, docstring) | ⬜ |
| 6 | 한계 인정 및 에스컬레이션 (3회 실패 → 보고) | ⬜ |
| 7 | 점진적 수집 전략 (10건 → 100건 → 전체) | ⬜ |

Phase 1에서는 원칙 1, 2만 체험했습니다. 나머지 5개는 **시스템 프롬프트의 상세한 지침**으로 에이전트에게 주입합니다.

또한 Coder의 **핵심 역할 맥락**은:
- Navigator가 설계한 **Blueprint(크롤링 청사진)** 를 받아서
- `rendering_type`에 따라 **requests/BS4** 또는 **playwright** 코드를 자동 생성
- `selectors` 딕셔너리의 CSS 셀렉터를 코드에 적용
- 생성한 코드를 실행하고 **수집된 데이터의 품질을 검증**

---
### Part 4: 도구(Tools) 설계

#### 📂 `app/tools/coder.py`

v1 Coder는 **6개의 전문 도구**를 사용합니다. Phase 1에서 만든 4개에 2개가 추가됩니다:

| 도구 | Phase 1 | v1 추가 | 역할 |
|:---|:---:|:---:|:---|
| `read_code_file` | ✅ | | 줄 번호와 함께 코드 읽기 |
| `edit_code_file` | ✅ | | 특정 줄만 외과수술적 교체 |
| `create_new_file` | ✅ | | 새 파일 생성 |
| `run_python_script` | ✅ | | 파이썬 스크립트 실행 |
| `write_text_file` | | 🆕 | JSON/CSV/MD 등 텍스트 문서 작성 |
| `validate_collected_data` | | 🆕 | 수집된 JSON 데이터 품질 검증 |

> 📖 `@tool(parse_docstring=True)`로 Docstring이 LLM이 이해하는 스키마로 자동 변환됩니다.

---

#### `ARTIFACT_DIR` — 공통 상수

`app/tools/common.py`에서 프로젝트 루트 기준으로 `artifacts/code` 경로를 설정합니다.
Phase 1의 `code_artifacts/`와 같은 원리입니다.

In [ ]:
# 📂 이 코드는 app/tools/common.py에 있습니다.
import os

# app/tools/common.py에서는 __file__ 기준으로 프로젝트 루트를 찾지만,
# 노트북에서는 환경변수를 사용합니다.
_PROJECT_ROOT = os.getenv("PROJECT_ROOT", os.getcwd())
ARTIFACT_DIR_V1 = os.path.join(_PROJECT_ROOT, "artifacts", "code")
os.makedirs(ARTIFACT_DIR_V1, exist_ok=True)

print(f"📁 v1 Artifact 디렉토리: {ARTIFACT_DIR_V1}")

#### 도구 1~4: 코드 파일 관리 도구

Phase 1에서 만든 4개 도구의 `app/` 버전입니다. 로직은 같지만 더 간결하게 작성되어 있습니다.

In [ ]:
# 📂 이 코드는 app/tools/coder.py에 있습니다.
import json
import subprocess
from langchain.tools import tool

@tool(parse_docstring=True)
def read_code_file_v1(filepath: str, start_line: int = 1, end_line: int = None) -> str:
    """파일 내용을 줄 번호와 함께 읽어옵니다.
    
    Args:
        filepath: 읽을 파일명 (artifacts/code 폴더 기준)
        start_line: 시작 줄
        end_line: 끝 줄
    """
    sf = os.path.join(ARTIFACT_DIR_V1, os.path.basename(filepath))
    if not os.path.exists(sf): return "[Error] 존재하지 않는 파일입니다."
    with open(sf, "r", encoding="utf-8") as f: lines = f.readlines()
    end = end_line if end_line else len(lines)
    start = max(1, start_line)
    return "\n".join([f"{i+1:03d} | {lines[i].rstrip()}" for i in range(start-1, min(end, len(lines)))])

@tool(parse_docstring=True)
def edit_code_file_v1(filepath: str, start_line: int, end_line: int, new_content: str) -> str:
    """기존 파이썬 파일의 특정 라인 구간만 외과수술처럼 교체합니다.
    
    Args:
        filepath: 수정할 파일명
        start_line: 시작 줄
        end_line: 교체가 완료될 줄 번호
        new_content: 새로 삽입될 코드 덩어리
    """
    sf = os.path.join(ARTIFACT_DIR_V1, os.path.basename(filepath))
    with open(sf, "r", encoding="utf-8") as f: lines = f.readlines()
    nl = [l+"\n" for l in new_content.split("\n")]
    lines = lines[:start_line-1] + nl + lines[end_line:]
    with open(sf, "w", encoding="utf-8") as f: f.writelines(lines)
    return f"[Success] {filepath} 교체 완료"

@tool(parse_docstring=True)
def create_new_file_v1(filepath: str, content: str) -> str:
    """파이썬 또는 텍스트 파일을 새로 완전히 생성합니다.
    
    Args:
       filepath: 파일명
       content: 담길 코드 원문
    """
    with open(os.path.join(ARTIFACT_DIR_V1, os.path.basename(filepath)), "w", encoding="utf-8") as f:
        f.write(content)
    return f"[Success] {filepath} 생성"

print("✅ 도구 1~3 정의 완료: read_code_file, edit_code_file, create_new_file")

#### 도구 4: `write_text_file` — 텍스트 문서 작성

`create_new_file`과 기능은 동일하지만, **JSON, Markdown, CSV** 등 텍스트 문서용으로 분리되어 있습니다.
LLM이 도구 이름과 Docstring으로 "코드 파일"과 "데이터 파일"을 구분하도록 설계되어 있습니다.

In [ ]:
# 📂 이 코드는 app/tools/coder.py에 있습니다.
@tool(parse_docstring=True)
def write_text_file(filepath: str, content: str) -> str:
    """JSON, Markdown, CSV 등 텍스트 문서를 작성합니다 (create_new_file 동일 스펙).
    
    Args:
        filepath: 파일명
        content: 파일에 기록할 텍스트 내용
    """
    return create_new_file_v1.invoke({"filepath":filepath, "content":content})

print("✅ 도구 4 정의 완료: write_text_file")

#### 도구 5: `run_python_script` (app/ 버전)

In [ ]:
# 📂 이 코드는 app/tools/coder.py에 있습니다.
@tool(parse_docstring=True)
def run_python_script_v1(filepath: str, script_args: str = "") -> str:
    """작성한 특정 파이썬 코드를 로컬 환경에서 비동기가 아닌 서브프로세스로 실행해 봅니다.
    
    Args:
       filepath: 파일 명
       script_args: CLI 어규먼트
    """
    sf = os.path.basename(filepath)
    cmd = ["python", sf]
    if script_args: cmd.extend(script_args.split())
    
    try:
        res = subprocess.run(cmd, cwd=ARTIFACT_DIR_V1, capture_output=True, text=True, timeout=120)
        out = res.stdout
        if res.stderr: out += f"\n[Error Output]\n{res.stderr}"
        return out if out.strip() else "[시스템] 실행되었으나 표준 출력이 없습니다."
    except Exception as e:
        return f"[시스템 에러] {e}"

print("✅ 도구 5 정의 완료: run_python_script")

#### 도구 6: `validate_collected_data` — 수집 데이터 품질 검증 🆕

이 도구는 Phase 1에는 없었던 **v1 전용 도구**입니다.

웹 크롤링 후 생성된 JSON 파일의 **데이터 품질을 자동으로 체크**합니다:
- 총 수집 건수
- 필드별 빈 값(empty value) 수
- 최상위 구조 검증 (배열인지 여부)

```
📊 총 25건 수집
   필드 'title'의 빈 값 수: 0
   필드 'price'의 빈 값 수: 3    ← 문제 발견!
   필드 'link'의 빈 값 수: 0
```

> 💡 이 도구 덕분에 Coder는 "코드가 실행되었다"와 "데이터가 올바르게 수집되었다"를 구분할 수 있습니다.

In [ ]:
# 📂 이 코드는 app/tools/coder.py에 있습니다.
@tool(parse_docstring=True)
def validate_collected_data(filepath: str) -> str:
    """수집된 JSON 데이터 파일(리스트 딕셔너리 구조)의 품질 및 정합성을 체크합니다.
    
    Args:
        filepath: 검증할 JSON 파일명
    """
    if os.path.exists(filepath):
        sf = filepath
    elif os.path.exists(os.path.join(ARTIFACT_DIR_V1, os.path.basename(filepath))):
        sf = os.path.join(ARTIFACT_DIR_V1, os.path.basename(filepath))
    else:
        sf = filepath
        
    try:
        with open(sf, 'r', encoding='utf-8') as f: data = json.load(f)
    except Exception as e: 
        return f"[Error] 파일 읽기 또는 파싱 불가: {e}"
    
    if not isinstance(data, list): return "[Info] 최상위가 배열이 아닙니다."
    if not data: return "[Warning] 수집된 데이터가 0건입니다!"
    
    fields = list(data[0].keys())
    rep = [f"📊 총 {len(data)}건 수집"]
    for field in fields:
        empty = sum(1 for d in data if not d.get(field))
        rep.append(f"   필드 '{field}'의 빈 값 수: {empty}")
    return "\n".join(rep)

print("✅ 도구 6 정의 완료: validate_collected_data")

#### 🧪 독립 실행 테스트: `validate_collected_data`

샘플 JSON 파일을 만들어 검증 도구를 직접 실행해봅시다.

In [ ]:
# 샘플 JSON 데이터 생성
sample_data = [
    {"title": "Python 입문", "author": "김개발", "price": "15000"},
    {"title": "자바 완성", "author": "", "price": "22000"},
    {"title": "", "author": "이코딩", "price": "18000"},
    {"title": "웹 크롤링", "author": "박데이터", "price": ""},
]

sample_path = os.path.join(ARTIFACT_DIR_V1, "sample_books.json")
with open(sample_path, "w", encoding="utf-8") as f:
    json.dump(sample_data, f, ensure_ascii=False, indent=2)

# validate_collected_data 도구 실행
result = validate_collected_data.invoke({"filepath": "sample_books.json"})
print(result)

---
### Part 5: 프롬프트(Prompts) — Coder의 두뇌 설계

#### 📂 `app/prompts/coder.py`

Phase 1의 5줄짜리 프롬프트와 v1의 전문 프롬프트를 비교해봅시다:

| | Phase 1 프롬프트 | v1 프롬프트 |
|:---|:---|:---|
| **길이** | ~5줄 | ~70줄 |
| **역할 정의** | "코딩 비서" | "최고 수준의 시니어 파이썬 SWE" |
| **행동 지침** | 4개 (기본) | 7개 (시니어 엔지니어링) + 4개 (스크래핑) |
| **에러 처리** | "다시 생성" | "read → edit 수술, 3회 반복 시 에스컬레이션" |
| **데이터 검증** | 없음 | validate → 점진적 수집 (10건 → 전체) |
| **Blueprint 해석** | 없음 | rendering_type, anti_bot_notes 판단 |

`CODER_SYSTEM_PROMPT`는 두 가지 섹션으로 구성됩니다:

1. **시니어 엔지니어링 행동 지침** (7개) — 범용 코딩 원칙
2. **웹 스크래핑 핵심 행동 지침** (4개) — Blueprint→코드 변환 전문 지침

> 💡 **프롬프트는 에이전트의 행동을 결정하는 가장 강력한 레버**입니다. Phase 1의 도구와 v1의 도구는 동일하지만, 프롬프트의 차이만으로 에이전트의 품질이 극적으로 달라집니다.

In [ ]:
# 📂 이 코드는 app/prompts/coder.py에 있습니다.
CODER_SYSTEM_PROMPT = """
당신은 최고 수준의 시니어 파이썬 소프트웨어 엔지니어(Senior SWE)입니다.
당신의 임무는 요구사항에 맞춰 코드를 견고하게 설계, 작성, 테스트, 그리고 스스로 디버깅하여 오류 없이 작동하도록 완성하는 것입니다.

[시니어 엔지니어링 행동 지침]
1. 분리된 기능의 영리한 사용:
   - 이전처럼 코드를 쓰자마자 자동으로 실행되지 않습니다.
   - 당신에게는 "코드 저장(create/edit)" 권한과 "코드 실행(run)" 권한이 완전히 분리된 4개의 도구가 주어집니다. 이를 적재적소에 사용하세요.

2. 정밀한 외과 수술적 수정 (Surgical Edit):
   - 파일을 처음 만들 때는 `create_new_file`을 쓰세요.
   - 단, 기존 파일에 에러가 났거나 기능을 덧붙일 때는 절대로 전체 코드를 다시 작성하지 마세요.
   - 반드시 `read_code_file`을 호출해 코드를 라인 번호와 함께 읽어들인 뒤, 에러가 발생한 지점을 찾아내고 `edit_code_file`을 이용해 특정 라인(start_line~end_line)만 아주 타겟팅하여 교체하세요.

3. 검증 없는 코딩은 없다 (Test-Driven):
   - 코드를 생성했거나 특정 라인을 수정(edit)했다면, 머리로 생각한 대로 돌아갈 것이라 오만하게 확신하지 마세요.
   - 반드시 그 직후에 `run_python_script` 툴을 써서 파이썬 파일을 터미널에서 실행해봐야 합니다.
   - 실행 결과에 붉은색 [Error Output]이 잡히거나 무한 루프에 빠진다면, 당황하지 말고 에러 메시지와 줄 번호(Line number)를 분석하여 위 2번 지침(수술적 수정) 과정을 즉시 반복하여 디버깅하세요.

4. 젠틀한 소통:
   - 뒤에서 수많은 수정/실행/디버깅 시행착오를 겪었더라도, 최종적으로 유저에게는 "어떻게 접근해서 어떻게 문제를 해결했는지", "최종 실행 결과는 무엇인지" 깔끔하게 보고하세요.
   - 유저가 물어보는 사항에 대해 친절하고 명료하게 답변하세요.

5. 코드 품질 기준 (Code Quality):
   - 작성하는 모든 파이썬 코드는 PEP 8 스타일(들여쓰기 4칸, 변수명 snake_case 등)을 준수하세요.
   - 모든 함수와 클래스에는 반드시 한 줄 docstring을 작성하세요. (예: 함수 첫 줄에 기능을 한 문장으로 설명하는 문자열 리터럴)

6. 한계 인정 및 에스컬레이션 (Error Escalation):
   - 동일한 에러가 3회 이상 반복되면 스스로 고치려는 시도를 즉시 중단하세요.
   - 대신 다음 내용을 유저에게 명확히 보고하세요:
     1) 발생한 에러 메시지 원문
     2) 시도한 수정 방법 목록 (회차별)
     3) 해결하지 못한 이유에 대한 분석 및 유저에게 요청할 사항

7. 점진적 수집 전략 (Progressive Collection):
   - 대량 데이터를 수집하는 스크립트를 작성할 때, 처음부터 전체를 수집하지 마세요.
   - 먼저 10건만 수집하도록 제한하여 실행하고, 데이터가 올바른지 확인하세요.
   - 검증이 끝나면 수량을 늘려서(100건 → 전체) 단계적으로 확대하세요.
   - 이렇게 하면 셀렉터 오류나 사이트 차단을 조기에 발견할 수 있습니다.

[웹 스크래핑 핵심 행동 지침]

당신의 주요 임무는 다른 에이전트(Navigator)가 작성한 JSON 형태의 'Blueprint(크롤링 청사진)'를 건네받아, 
그 지시대로 완벽하게 작동하는 파이썬 크롤링 스크립트를 작성하고 실행하는 것입니다.

1. Blueprint 완벽 해석:
   - 주어진 JSON 데이터에서 `rendering_type`, `anti_bot_notes`, `layers` 등의 정보를 완벽하게 분석하세요.
   - 렌더링 방식이 "Static SSR"이면 `requests`와 `BeautifulSoup4`를 사용해 가볍게 작성하세요.
   - 렌더링 방식이 "Dynamic CSR/JS"이거나 `anti_bot_notes`에 JS 렌더링이 언급되어 있다면, `playwright`의 
     동기 방식(sync_playwright)을 사용하여 동적 코드를 작성하세요.
     (이 스크립트는 독립 프로세스에서 실행되므로 동기 방식이 더 안정적입니다.)

2. 오류 방어 및 안티봇 우회 (Robstness):
   - `anti_bot_notes`에 경고가 있다면, User-Agent 위조(fake_useragent), 브라우저 헤더 추가, 
     요청 간 대기(time.sleep 또는 asyncio.sleep) 코드를 반드시 포함하세요.
   - CSS 셀렉터를 사용할 때, 요소가 없을 경우에 대비해 항상 `try-except` 블럭 또는 `if element is None` 
     예외 처리를 견고하게 넣어야 합니다.

3. 코드 작성 방식:
   - `create_new_file` 도구를 사용해 스크립트 파일을 생성합니다. 파일명은 `naver_news_crawler.py` 처럼 
     직관적으로 지으세요.
   - 작성된 스크립트의 맨 아래에는 `if __name__ == "__main__":` 블록을 만들고 직접 함수를 실행하는 코드를 넣으세요.
   - 파싱한 정보(예: 기사 제목 5개 등)는 화면에 `print`로 보기 좋게 출력되도록 하세요. 
     (그래야 run_python_script 툴로 당신이 결과를 볼 수 있습니다!)

4. 검증 및 디버깅:
   - 스크립트를 생성한 직후, 반드시 `run_python_script` 툴을 사용해 자신이 만든 파이썬 코드를 실행하세요.
   - 실행 후 JSON 파일이 생성되었다면, 즉시 `validate_collected_data` 툴로 데이터 품질을 검증하세요.
   - 점진적 수집과 연계한 워크플로우: 10건 테스트 → run → validate → 문제없으면 수량 확대 → run → validate
   - 만약 에러[Error Output]가 떨어지거나 아무것도 수집되지 않는다면, 
     `read_code_file` -> `edit_code_file` 콤보를 사용해 에러가 난 라인만 수술하듯 고치고 다시 실행하세요.

"""

print(f"✅ CODER_SYSTEM_PROMPT 로드 완료 ({len(CODER_SYSTEM_PROMPT)}자)")

---
### Part 6: 스키마(Schemas) — SeniorCoderContext

#### 📂 `app/schemas.py`

Navigator의 `NavigatorContext`에는 `shared_browser`와 `response_mode`가 있었습니다.
반면 Coder의 `SeniorCoderContext`는 **현재 비어있는 dataclass**입니다.

**"왜 비어있어도 context_schema를 지정하는가?"**

1. **일관된 API**: 모든 에이전트가 동일한 `context_schema` 패턴을 사용하면 팀 규약이 통일됨
2. **향후 확장성**: 디버깅 모드 플래그, 작업 디렉토리 경로, 최대 재시도 횟수 등을 추가하기 쉬움
3. **서빙 시 주입**: 서버에서 에이전트를 호출할 때 런타임 정보를 주입하는 통로

```python
# 미래에 이렇게 확장 가능:
@dataclass
class SeniorCoderContext:
    debug_mode: bool = False
    max_retries: int = 3
    working_dir: str = "artifacts/code"
```

In [ ]:
# 📂 이 코드는 app/schemas.py에 있습니다.
from dataclasses import dataclass

@dataclass
class SeniorCoderContext:
    pass

print("✅ SeniorCoderContext 정의 완료")

---
### Part 7: 미들웨어(Middleware) — FilesystemFileSearchMiddleware

#### LangChain 내장 미들웨어

개발자가 코드를 짜려면 **현재 작업 폴더에 어떤 파일들이 있는지** 파악해야 합니다.

Navigator에서는 `dynamic_response_format`이라는 **커스텀 미들웨어**(`@wrap_model_call`)를 만들었습니다.
Coder에서는 LangChain에 내장된 **`FilesystemFileSearchMiddleware`**를 사용합니다.

이 미들웨어는 에이전트에게 **파일 시스템 검색/읽기 도구를 자동으로 추가**합니다:
- 지정된 `root_path` 아래의 파일 목록 조회
- 파일 내용 검색 (ripgrep 지원)
- 에이전트가 작업 폴더의 맥락을 파악하고 관련 파일을 참조 가능

| Navigator 미들웨어 | Coder 미들웨어 |
|:---|:---|
| 커스텀 (`@wrap_model_call`) | 내장 (`FilesystemFileSearchMiddleware`) |
| 응답 포맷 동적 전환 | 파일 검색/읽기 도구 자동 추가 |
| 직접 구현 필요 | 설정만으로 사용 가능 |

> 📖 미들웨어에 대한 자세한 설명은 `reference/LangChain/middleware.md`를 참고하세요.

In [ ]:
# 📂 이 코드는 app/agents/coder.py에서 사용됩니다.
from langchain.agents.middleware import FilesystemFileSearchMiddleware

# artifacts 폴더 전체 컨텍스트용 검색 미들웨어
# root_path의 상위 디렉토리를 지정하여 더 넓은 범위를 검색할 수도 있습니다.
test_root_dir = os.path.dirname(ARTIFACT_DIR_V1)

coder_middleware = [
    FilesystemFileSearchMiddleware(
        root_path=test_root_dir,
        use_ripgrep=True,
        max_file_size_mb=10,
    )
]

print(f"✅ FilesystemFileSearchMiddleware 설정 완료 (root: {test_root_dir})")

---
### Part 8: 최종 조립 — Coder 에이전트 완성

지금까지 이식한 모든 조각을 조립합니다:

| 구성요소 | 파트 | 실제 파일 |
|:---|:---|:---|
| 도구 6개 | Part 4 | `app/tools/coder.py` |
| 시스템 프롬프트 | Part 5 | `app/prompts/coder.py` |
| Context 스키마 | Part 6 | `app/schemas.py` |
| 미들웨어 | Part 7 | LangChain 내장 |

이 모든 것을 `create_agent()`로 조립하면 Coder 에이전트가 완성됩니다.

> 실제 프로젝트에서는 `from app.agents.coder import create_coder_agent`로 한 줄 임포트합니다.

In [ ]:
# 📂 이 코드는 app/agents/coder.py에 있습니다.
# 실제 프로젝트에서는 아래 한 줄로 임포트합니다:
# from app.agents.coder import create_coder_agent

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import FilesystemFileSearchMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def create_coder_agent(model_name: str = "google_genai:gemini-flash-latest", temperature: float = 0.2):
    """데이터 청사진을 코드로 제작/수행하는 Coder 에이전트 생성"""
    model = init_chat_model(model_name, temperature=temperature)
    checkpointer = InMemorySaver()
    
    # artifacts 폴더 전체 컨텍스트용 검색 미들웨어
    test_root_dir = os.path.dirname(ARTIFACT_DIR_V1)
    
    middleware = [
        FilesystemFileSearchMiddleware(
            root_path=test_root_dir,
            use_ripgrep=True,
            max_file_size_mb=10,
        )
    ]
    
    agent = create_agent(
        model=model,
        system_prompt=CODER_SYSTEM_PROMPT,           # Part 5
        context_schema=SeniorCoderContext,            # Part 6
        tools=[read_code_file_v1,                     # Part 4
               edit_code_file_v1,                     # Part 4
               create_new_file_v1,                    # Part 4
               write_text_file,                       # Part 4
               run_python_script_v1,                  # Part 4
               validate_collected_data],              # Part 4
        checkpointer=checkpointer,
        middleware=middleware                          # Part 7
    )
    return agent

print("✅ create_coder_agent 팩토리 함수 정의 완료")

#### 🧪 테스트 1: 자율 디버깅 능력 검증

Phase 1과 동일한 3단계 미션을 v1 Coder에게 줍니다.
시니어 프롬프트가 적용된 v1 Coder가 **더 체계적으로 작업**하는지 비교해보세요.

관찰 포인트:
- PEP 8 스타일, docstring 작성 여부 (원칙 5)
- create → run → read → edit → run 루프 사용 (원칙 2, 3)
- 최종 보고의 깔끔함 (원칙 4)

In [ ]:
# v1 Coder 에이전트 생성
coder = create_coder_agent()

config_test1 = {"configurable": {"thread_id": "coder_v1_test_1"}}
context_test1 = SeniorCoderContext()

# 3단계 미션 (Phase 1과 동일)
query = (
    "1. 'calculator.py'를 파이썬으로 만들어줘. "
    "add, subtract 함수를 가진 Calculator 클래스를 짜고 "
    "출력으로 'Calculator Created'를 찍게 한 뒤 실행해봐.\n"
    "2. 실행을 확인한 뒤에는 calculator.py의 특정 라인을 수정(edit_code_file 사용)해서 "
    "multiply와 divide 함수를 추가해 봐. 전체 파일을 새로 덮어쓰면 절대 안 돼!\n"
    "3. 마지막으로 수정된 파일을 다시 한번 실행해보고 오류가 없으면 결과를 보고해줘."
)
print(f"👤 User Mission:\n{query}\n")
print("-" * 50)

response = await coder.ainvoke(
    {"messages": [HumanMessage(query)]},
    config=config_test1,
    context=context_test1
)

print(f"\n🤖 Senior Coder: {response['messages'][-1].content}")

In [ ]:
for m in response["messages"]:
    m.pretty_print()

#### 🧪 테스트 2: 멀티턴 대화 — 파일 확인

이전 세션을 이어서, Coder에게 방금 만든 파일의 내용을 확인해달라고 요청합니다.

In [ ]:
query2 = "좋아, 그럼 방금 생성한 코드 파일 안에 무슨 내용이 적혀있는지 정확히 확인하고 알려줄래?"
print(f"👤 User: {query2}\n")
print("-" * 50)

response2 = await coder.ainvoke(
    {"messages": [HumanMessage(query2)]},
    config=config_test1,
    context=context_test1
)
print(f"\n🤖 Senior Coder: {response2['messages'][-1].content}")

In [ ]:
for m in response2["messages"]:
    m.pretty_print()

---
### 📦 정리: Coder v1의 모듈 구조

지금까지 이식한 코드가 `app/` 프로젝트에서 어떻게 모듈화되어 있는지 정리합니다.

```
app/
├── agents/
│   └── coder.py         ← Part 8: create_coder_agent() 팩토리 함수
├── tools/
│   ├── coder.py          ← Part 4: 도구 6개 (read/edit/create/write/run/validate)
│   └── common.py         ← ARTIFACT_DIR 상수, CostTracker 유틸리티
├── prompts/
│   └── coder.py          ← Part 5: CODER_SYSTEM_PROMPT
└── schemas.py            ← Part 6: SeniorCoderContext
```

### 🔜 다음 노트북

**05번 노트북**에서는 **Navigator와 Coder를 연결하는 Supervisor 에이전트**를 만들어,
완전한 멀티에이전트 파이프라인을 완성합니다.

```
사용자 요청 → Supervisor → Navigator (Blueprint 설계)
                         → Coder (코드 생성/실행/검증)
                         → 결과 보고
```

Navigator가 설계한 "청사진"을 Coder가 "코드"로 바꾸는 것 — 이것이 멀티 에이전트 파이프라인의 핵심입니다.